# Minimum safety factor - verification

Use the **Python (FAITH labelmaker)** kernel. Set `shot`, drag a time range on
any panel, then press *Mark present* / *Mark absent*, *Verify* and *Save*.

`qmin` is the offline EFIT01 `aeqdsk:qmin` standing in for `qmin_EFITRT2`, and
every shot in this category's roster has it. The classes are the `qmin_rule`
ones: 0 absent, 1 low, 2 hybrid, 3 elevated, 4 high, with thresholds at
qmin > 0.95 (hybrid), > 1.5 (elevated) and > 2 (high) - the three dashed lines
on the first panel.

The second panel's vertical axis is **normalized psi, not rho**, despite the
feature store calling its radial grid `RHO_GRID`: the geqdsk `qpsi` node
arrives on 65 points of normalized psi and the archive keeps their even
indices (`features/namespace.py` says so in capitals). psi_n = 0.5 is roughly
rho = 0.7, so do not read a rational surface's radial position off it against
the label row below, which genuinely is in rho.

What to check: that the flat-top is where the rule says it is, and that a
class change sits on a real qmin crossing rather than on an EFIT excursion of
one or two frames.


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np

from labeler.config import Paths
from labeler.events.verify import Panel, review
from labeler.features.store import read_feature

event = "minimum_safety_factor"
shot = 1  # replace with a shot from shots.csv
source = "extend_qmin_rule/recommender_v1"

features = Paths.from_env().features_file(shot)

qmin = read_feature(features, "qmin")
qpsi = read_feature(features, "qpsi")
ip = read_feature(features, "ip")

panels = [
    Panel(
        title="qmin (EFIT01 aeqdsk)",
        x=qmin.x * 1000.0,
        y=qmin.y,
        ylabel="q",
        # The rule's class thresholds, as three dashed lines. A `bands`
        # entry 0.01 tall at 8% opacity, on an axis spanning ~0.8-3, is
        # invisible; a threshold is a line, so `hlines` draws one.
        hlines=[0.95, 1.5, 2.0],
    ),
    Panel(
        title="q profile",
        kind="heatmap",
        x=qpsi.x * 1000.0,
        y=np.linspace(0.0, 1.0, qpsi.y.shape[0]),
        z=qpsi.y,
        # normalized psi, NOT rho - see the header, and
        # `features/namespace.py`'s note on the qpsi radial axis
        ylabel="psi_n",
    ),
    Panel(title="ip", x=ip.x * 1000.0, y=ip.y, ylabel="A"),
]

In [ ]:
session = review(event, shot, panels, source=source)
session

`category` on a correction is the q-min class, not a binary flag:
0 absent, 1 low, 2 hybrid, 3 elevated, 4 high. *Mark present* writes 1, so for
any other class call `session.mark(t_start, t_end, category=3)` directly.

```python
from labeler.events.verify import read_corrections, review_path

read_corrections(review_path(event, shot))
```
